# Week 3a.1 — Short-term Memory

Our agents so far have amnesia. Every `invoke` starts from nothing: the model does not remember what you asked one cell ago. Today we fix that for a single conversation. Memory across conversations and sessions is a different problem, which we take up later in the course.

In [ ]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The problem

In [ ]:
response = model.invoke("Hi, my name is Alex and I am training for a marathon in November.")
print(response.text)

In [ ]:
response = model.invoke("What am I training for?")
print(response.text)

The model has no idea. It is **stateless**: nothing carries over from one call to the next. What looked like a conversation in a chat app was never memory inside the model; the application was resending the history every time.

We already have the tool for this: a call can take a **list of messages**. That list is the memory.

## 2. Memory by hand

Keep a list. Append every message, human and AI, and send the whole list on every call.

In [ ]:
from langchain.messages import HumanMessage

history = []

history.append(HumanMessage(content="Hi, my name is Alex and I am training for a marathon in November."))
response = model.invoke(history)
history.append(response)

print(response.text)

In [ ]:
history.append(HumanMessage(content="What am I training for?"))
response = model.invoke(history)
history.append(response)

print(response.text)

In [ ]:
#TODO: ask one more follow-up that only makes sense with memory
# (for example: "How many weeks do I have left?"). Follow the same
# append, invoke, append pattern.


That is all short-term memory is: the application rereads the entire conversation to the model on every single call. Nothing is stored inside the model.

## 3. Agents with threads

For agents, LangChain packages this pattern so you do not manage the list yourself. Give `create_agent` a **checkpointer** and it saves the conversation state after every call, filed under a `thread_id` you choose. Same thread, same memory; new thread, blank slate.

- https://docs.langchain.com/oss/python/langchain/short-term-memory

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You're a helpful assistant who answers users' questions concisely.",
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "alex"}}

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="Hi, my name is Alex and I am training for a marathon in November.")]}, config)
print(result["messages"][-1].text)

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="What am I training for?")]}, config)
print(result["messages"][-1].text)

Same question, different thread:

In [ ]:
#TODO: invoke the agent with the same follow-up question but a new
# thread_id (for example "jordan") and see what changes.


One agent, many independent conversations, one line of difference. This is exactly how a support desk serves many customers at once, which is Friday's project.

## 4. Memory has a cost

The history grows on every turn, and the model rereads all of it on every call. You pay for that twice: context windows are finite, and tokens cost money.

In [ ]:
result = agent.invoke({"messages": [HumanMessage(content="Suggest a training plan for this week.")]}, config)

print(len(result["messages"]), "messages in the thread")

In [ ]:
from langchain.messages import trim_messages

# Keep only the most recent messages. Here token_counter=len counts
# messages; production systems count actual tokens.
trimmed = trim_messages(
    result["messages"],
    strategy="last",
    token_counter=len,
    max_tokens=4,
)

for m in trimmed:
    print(type(m).__name__, '-', m.text[:60])

Trimming is the bluntest instrument: whatever falls off the end is gone, even if it was the customer's name. Smarter strategies, such as summarizing old turns or moving facts to external storage, are exactly where the course goes next week.

## 5. A chat loop

With threads, a real chat interface is a few lines. Uncomment and run; type `quit` to stop.

In [ ]:
# while True:
#     user = input("You: ")
#     if user.lower() in {"quit", "exit"}:
#         break
#     result = agent.invoke({"messages": [HumanMessage(content=user)]}, config)
#     print("Assistant:", result["messages"][-1].text)